# Final extras: CORAL domain adaptation and grayscale-augmentation training

Two items the instructor asked for that need Colab:

* **Part A, CORAL** (about 5 minutes, works on CPU). Aligns the WikiArt training features to each museum's feature statistics (Sun et al., 2016), then retrains the linear probe. Uses the CLIP, DINOv2 and SigLIP feature files already on Drive (`wikiart_runs/probes/`). No labels from the museums are used, only their images' features.
* **Part B, grayscale augmentation** (about 1 hour, needs a T4 GPU). Retrains the ResNet-50 baseline for seeds 1, 2, 3 with 20% of WikiArt training images turned gray (`src/train.py --grayscale-aug 0.2`), then scores the normal and the grayscale test images.

**How to run:** Runtime → Change runtime type → **T4 GPU** → Save, then **Runtime → Run all**. If there is no GPU, Part A still runs and Part B is skipped. If Colab disconnects, Run all again: finished seeds are skipped.

Results go to `wikiart_runs/final_extras/` next to `colab_data.zip`. The last cell of each part prints a short report.

## 0. Connect Drive and find the project folder

In [ ]:
import os, sys, time, json, urllib.request
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
CANDIDATES = ['/content/drive/MyDrive/wikiart-to-museum', '/content/drive/MyDrive']
BASE = next((c for c in CANDIDATES if os.path.exists(f'{c}/wikiart_runs/probes/features_clip.npz')), None)
if BASE is None:
    raise SystemExit('wikiart_runs/probes/features_clip.npz not found in My Drive or My Drive/wikiart-to-museum. '
                     'Run notebooks/foundation_probes.ipynb first, or add the shared folder to My Drive.')
PROBES = f'{BASE}/wikiart_runs/probes'
OUT = f'{BASE}/wikiart_runs/final_extras'; os.makedirs(OUT, exist_ok=True)
print('Features:', PROBES); print('Saves   :', OUT)

RAW = 'https://raw.githubusercontent.com/abdull254/wikiart-to-museum/main/'
os.makedirs('/content/repo', exist_ok=True)
for f in ('data/final/wikiart_split.csv', 'data/final/museum_test.csv'):
    dst = f'/content/repo/{os.path.basename(f)}'
    if not os.path.exists(dst):
        urllib.request.urlretrieve(RAW + f, dst)
split = pd.read_csv('/content/repo/wikiart_split.csv')
museum = pd.read_csv('/content/repo/museum_test.csv')
CLASSES = sorted(split['style'].unique()); C = len(CLASSES)
museum = museum[museum['style'].isin(CLASSES)]
sets = {s: split[split['split'] == s].reset_index(drop=True) for s in ('train', 'val', 'test')}
sets['met'] = museum[museum['museum'] == 'met'].reset_index(drop=True)
sets['rijks'] = museum[museum['museum'] == 'rijks'].reset_index(drop=True)
Y = {k: np.array([CLASSES.index(s) for s in df['style']]) for k, df in sets.items()}
ARTIST = {'test': ('w:' + sets['test']['artist_key'].astype(str)).values}
for k in ('met', 'rijks'):
    a = sets[k]['artist'].fillna('').astype(str)
    ARTIST[k] = np.where(a.str.strip() == '', 'anon:' + sets[k]['id'].astype(str), 'm:' + a)
for k, df in sets.items(): print(f'{k:<6} {len(df):>6}')

## Part A. CORAL on frozen features (CPU is fine)

For each model and each museum:
1. L2-normalize the features and standardize them with WikiArt-train statistics (as in the probe notebook).
2. **No adaptation:** logistic regression on WikiArt train, C chosen on WikiArt val (same as the probes).
3. **CORAL:** whiten the WikiArt train features with their own covariance and re-color them with the museum's covariance (`Xs · Cs^-1/2 · Ct^1/2`, both covariances + identity), re-center on the museum mean, then fit the same classifier with the same C. The museum covariance uses all that museum's test images (no labels), the standard transductive CORAL setting.
4. Score the museum's **unseen-artist** paintings and compare with a painter-level paired bootstrap.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, f1_score

def mat_pow(Cm, p):
    w, V = np.linalg.eigh(Cm); w = np.clip(w, 1e-6, None)
    return (V * w ** p) @ V.T

def coral(Xs, Xt):
    Cs = np.cov(Xs, rowvar=False) + np.eye(Xs.shape[1])
    Ct = np.cov(Xt, rowvar=False) + np.eye(Xt.shape[1])
    return (Xs - Xs.mean(0)) @ mat_pow(Cs, -0.5) @ mat_pow(Ct, 0.5) + Xt.mean(0)

def bal(y, p): return 100 * balanced_accuracy_score(y, p)
def paired_boot(y, pa, pb, art, B=2000, seed=0):
    ids, a = np.unique(art, return_inverse=True); A = len(ids)
    N = np.zeros((A, C)); np.add.at(N, (a, y), 1)
    KA = np.zeros((A, C)); np.add.at(KA, (a, y), pa == y)
    KB = np.zeros((A, C)); np.add.at(KB, (a, y), pb == y)
    W = np.random.default_rng(seed).multinomial(A, np.ones(A) / A, size=B)
    n = W @ N; ok = n > 0
    ra = np.nanmean(np.where(ok, (W @ KA) / np.where(ok, n, 1), np.nan), 1)
    rb = np.nanmean(np.where(ok, (W @ KB) / np.where(ok, n, 1), np.nan), 1)
    d = 100 * (ra - rb)
    return np.percentile(d, 2.5), np.percentile(d, 97.5), min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))

rows, lines = [], []
for name in ('clip', 'dinov2', 'siglip'):
    f = f'{PROBES}/features_{name}.npz'
    if not os.path.exists(f):
        print(name, ': no feature file, skipped'); continue
    F = dict(np.load(f))
    if not all(k in F and len(F[k]) == len(sets[k]) for k in sets):
        print(name, ': feature file does not match the sets, skipped', {k: len(F.get(k, [])) for k in sets}); continue
    F = {k: v / np.linalg.norm(v, axis=1, keepdims=True) for k, v in F.items()}
    sc = StandardScaler().fit(F['train']); F = {k: sc.transform(v).astype(np.float64) for k, v in F.items()}
    best = None
    for Cc in (0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0):          # chosen on WikiArt val only
        clf = LogisticRegression(C=Cc, max_iter=3000).fit(F['train'], Y['train'])
        b = bal(Y['val'], clf.predict(F['val']))
        if best is None or b > best[0]: best = (b, Cc, clf)
    Cc, base_clf = best[1], best[2]
    print(f'{name}: C = {Cc} (WikiArt val bal acc {best[0]:.1f})', flush=True)
    for k in ('met', 'rijks'):
        Xs = coral(F['train'], F[k])
        cor_clf = LogisticRegression(C=Cc, max_iter=3000).fit(Xs, Y['train'])
        unseen = (sets[k]['artist_group'] == 'unseen').values
        y = Y[k][unseen]; pb = base_clf.predict(F[k])[unseen]; pc = cor_clf.predict(F[k])[unseen]
        lo, hi, p = paired_boot(y, pc, pb, ARTIST[k][unseen])
        r = dict(model=name, museum=k, n=int(unseen.sum()), C=Cc,
                 bal_no_adapt=bal(y, pb), bal_coral=bal(y, pc), gain=bal(y, pc) - bal(y, pb), ci_low=lo, ci_high=hi, p=p,
                 f1_no_adapt=100 * f1_score(y, pb, average='macro'), f1_coral=100 * f1_score(y, pc, average='macro'),
                 top1_no_adapt=100 * (pb == y).mean(), top1_coral=100 * (pc == y).mean())
        for c in range(C):
            m = y == c
            r[f'recall_{CLASSES[c].split(" ")[0]}_no_adapt'] = 100 * (pb[m] == c).mean()
            r[f'recall_{CLASSES[c].split(" ")[0]}_coral'] = 100 * (pc[m] == c).mean()
        rows.append(r)
        print(f'  {k:<5} unseen n={r["n"]}: no adaptation {r["bal_no_adapt"]:.1f}  CORAL {r["bal_coral"]:.1f}  '
              f'gain {r["gain"]:+.1f} [{lo:+.1f}, {hi:+.1f}] p = {p:.3f}', flush=True)
coral_df = pd.DataFrame(rows)
coral_df.round(2).to_csv(f'{OUT}/coral_results.csv', index=False)
rep = ['CORAL on frozen features (Sun et al. 2016). Museum = unseen-artist paintings. Balanced accuracy in %.',
       'gain = CORAL minus no adaptation; painter-level paired bootstrap 95% CI (2,000).', '',
       coral_df[['model', 'museum', 'n', 'C', 'bal_no_adapt', 'bal_coral', 'gain', 'ci_low', 'ci_high', 'p',
                 'f1_no_adapt', 'f1_coral', 'top1_no_adapt', 'top1_coral']].round(2).to_string(index=False), '',
       'Recall per style (no adaptation -> CORAL):']
for r in rows:
    rep.append(f"  {r['model']:<7} {r['museum']:<5} " + '  '.join(
        f"{c.split(' ')[0][:9]} {r[f'recall_{c.split(chr(32))[0]}_no_adapt']:.0f}->{r[f'recall_{c.split(chr(32))[0]}_coral']:.0f}" for c in CLASSES))
open(f'{OUT}/coral_report.txt', 'w').write('\n'.join(rep) + '\n')
print('\n'.join(rep))

## Part B. Grayscale-augmentation training (T4 GPU, about 20 minutes per seed)

Identical to the baseline except `--grayscale-aug 0.2`: each WikiArt training image is turned gray with probability 0.2. Val, test and museum images are unchanged. After training, `src/evaluate.py` scores the normal (`base`) and the grayscale (`gray`) test images, to see whether the model now relies less on colour.

In [ ]:
import torch, shutil, glob, subprocess
HAS_GPU = torch.cuda.is_available()
if not HAS_GPU:
    print('No GPU: Part B skipped. (Runtime -> Change runtime type -> T4 GPU, then Run all.)')
else:
    print('GPU:', torch.cuda.get_device_name(0))
    ZIP = f'{BASE}/colab_data.zip'
    assert os.path.exists(ZIP), f'{ZIP} not found'
    if not os.path.exists('/content/data/final/wikiart_split.csv'):
        subprocess.run(f'cp "{ZIP}" /content/ && cd /content && unzip -q -o colab_data.zip && rm colab_data.zip', shell=True, check=True)
    for f in ('src/train.py', 'src/evaluate.py'):
        urllib.request.urlretrieve(RAW + f, f'/content/{os.path.basename(f)}')
    src = open('/content/train.py').read(); assert '--grayscale-aug' in src

In [ ]:
SEEDS = [1, 2, 3]
def finished(d):
    return os.path.exists(f'{d}/metrics.json') and os.path.exists(f'{d}/best.pt') and os.path.getsize(f'{d}/best.pt') > 10_000_000 \
        and os.path.exists(f'{d}/ablations.csv')
if HAS_GPU:
    os.chdir('/content')
    for SEED in SEEDS:
        name = f'resnet50_gray0.2_seed{SEED}'
        dst = f'{OUT}/{name}'
        if finished(dst):
            print(f'seed {SEED}: already on Drive, skipping'); continue
        t0 = time.time()
        if os.path.exists(f'/content/runs/{name}'): shutil.rmtree(f'/content/runs/{name}')
        print(f'\n===== seed {SEED}: training with grayscale augmentation =====', flush=True)
        subprocess.run(f'python /content/train.py --seed {SEED} --grayscale-aug 0.2 --eval-museums --workers 2 --name {name}',
                       shell=True, check=True)
        subprocess.run(f'python /content/evaluate.py --run /content/runs/{name} --variants base gray', shell=True, check=True)
        if os.path.exists(dst): shutil.move(dst, dst + time.strftime('_old_%Y%m%d_%H%M%S'))
        shutil.copytree(f'/content/runs/{name}', dst)
        assert finished(dst), 'copy to Drive failed: check Drive storage'
        print(f'seed {SEED}: saved in {(time.time() - t0) / 60:.0f} min')

In [ ]:
# Compare with the baseline (3 committed seeds) on the same paintings: balanced accuracy, unseen artists for museums
if HAS_GPU and all(finished(f'{OUT}/resnet50_gray0.2_seed{s}') for s in SEEDS):
    COMMIT = '5d10b4b'
    rows = []
    for s in SEEDS:
        bl = pd.read_csv(f'{RAW.replace("/main/", "/" + COMMIT + "/")}results/runs/resnet50_seed{s}/ablations.csv')
        ga = pd.read_csv(f'{OUT}/resnet50_gray0.2_seed{s}/ablations.csv')
        for v in ('base', 'gray'):
            for k in ('wikiart_test', 'met', 'rijks'):
                col = 'balanced_accuracy' if k == 'wikiart_test' else 'unseen_balanced_accuracy'
                b = bl[(bl.variant == v) & (bl.set == k)][col]; g = ga[(ga.variant == v) & (ga.set == k)][col]
                if len(b) and len(g):
                    rows.append(dict(seed=s, test_images=v, set=k, baseline=100 * b.iloc[0], gray_aug=100 * g.iloc[0]))
    t = pd.DataFrame(rows)
    from scipy import stats
    summ = t.groupby(['test_images', 'set']).agg(baseline=('baseline', 'mean'), baseline_sd=('baseline', 'std'),
                                                 gray_aug=('gray_aug', 'mean'), gray_aug_sd=('gray_aug', 'std')).reset_index()
    summ['diff'] = summ.gray_aug - summ.baseline
    summ['t_test_p'] = [stats.ttest_rel(g.gray_aug, g.baseline).pvalue for _, g in t.groupby(['test_images', 'set'])]
    rep = ['Grayscale augmentation (p = 0.2, training only) vs baseline ResNet-50, 3 seeds, balanced accuracy %.',
           'Museum rows = unseen artists. test_images: base = normal images, gray = grayscale test images.',
           'Colour reliance = base minus gray (smaller = relies less on colour).', '', summ.round(2).to_string(index=False), '']
    for m in ('baseline', 'gray_aug'):
        for k in ('wikiart_test', 'met', 'rijks'):
            b = summ[(summ.test_images == 'base') & (summ.set == k)][m].iloc[0]; g = summ[(summ.test_images == 'gray') & (summ.set == k)][m].iloc[0]
            rep.append(f'  colour reliance {m:<9} {k:<12} {b - g:5.1f} points')
    open(f'{OUT}/grayaug_report.txt', 'w').write('\n'.join(rep) + '\n')
    print('\n'.join(rep))
else:
    print('Part B not finished yet: Run all again (finished seeds are skipped).')